# Imports and Configuration

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib
import ipywidgets as widgets
from IPython.display import display, clear_output

matplotlib.rcParams["figure.dpi"] = 600

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


## Constants and Paths

Define shock data directory, default time deltas for upstream/downstream regions, and satellite-specific settings.

In [3]:
SHOCKS_DIR = Path("Data")
PARAMS_PATH = Path("Shocks/Shock Params.json")
DEFAULT_DTS = {"dt0_u": -5, "dt1_u": -2, "dt0_d": 2, "dt1_d": 5}

high_mag_density = {"ace", "dscovr"}
high_v_density = {"dscovr"}

# Shock Time Determination

Interactive shock parameter tuning for `Data` parquets.

## Data Loading Functions

Functions to load shock data from parquet files and manage the data cache.

In [4]:
def list_shock_dates():
    if not SHOCKS_DIR.exists():
        return []
    return sorted([p.name for p in SHOCKS_DIR.iterdir() if p.is_dir()])


def _ensure_datetime_index(df):
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for col in ["time", "Time", "datetime", "date", "timestamp", "Timestamp"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
            return df.set_index(col)
    df.index = pd.to_datetime(df.index)
    return df


def load_shock_folder(date):
    folder = SHOCKS_DIR / date
    dfs = {}
    for path in sorted(folder.glob("*.parquet")):
        sat = path.stem.lower()
        df = pd.read_parquet(path)
        df = _ensure_datetime_index(df)
        df = df.sort_index()
        # Remove duplicate indices, keeping first occurrence
        df = df[~df.index.duplicated(keep="first")]
        dfs[sat] = df
    return dfs


dfs_cache = {}


def get_dfs(date):
    if date not in dfs_cache:
        dfs_cache[date] = load_shock_folder(date)
    return dfs_cache[date]

## Parameter Management

Load, save, and manage shock parameters (t0, dt0_u, dt1_u, dt0_d, dt1_d) for each date and satellite combination.

In [5]:
def _seed_params_for_date(date, dfs):
    params = {}
    for sat, df in dfs.items():
        if df.empty:
            continue
        t0 = df.index[len(df) // 2]
        params[sat] = {"t0": t0, **DEFAULT_DTS}
    return params


def _parse_t0(value):
    if value is None:
        return None

    if isinstance(value, pd.Timestamp):
        return value

    if isinstance(value, (int, float, np.integer, np.floating)):
        # Legacy files may store epoch as ms (1e12) or ns (1e18).
        unit = "ns" if abs(float(value)) > 1e14 else "ms"
        return pd.to_datetime(int(value), unit=unit)

    if isinstance(value, str):
        try:
            return pd.Timestamp(value)
        except Exception:
            return None

    return None


def _normalize_sat_params(entry):
    if not isinstance(entry, dict):
        return None

    t0 = _parse_t0(entry.get("t0"))
    if t0 is None:
        return None

    return {
        "t0": t0,
        "dt0_u": int(entry.get("dt0_u", DEFAULT_DTS["dt0_u"])),
        "dt1_u": int(entry.get("dt1_u", DEFAULT_DTS["dt1_u"])),
        "dt0_d": int(entry.get("dt0_d", DEFAULT_DTS["dt0_d"])),
        "dt1_d": int(entry.get("dt1_d", DEFAULT_DTS["dt1_d"])),
    }


def _load_params():
    if not PARAMS_PATH.exists():
        return {}

    try:
        raw = json.loads(PARAMS_PATH.read_text())
    except Exception:
        return {}

    if not isinstance(raw, dict):
        return {}

    known_dates = set(list_shock_dates())
    out = {}

    for date_key, sats in raw.items():
        date = str(date_key)

        # Drop unknown / corrupted top-level keys to avoid cross-pollination.
        if date not in known_dates:
            continue

        if not isinstance(sats, dict):
            continue

        out[date] = {}
        for sat, entry in sats.items():
            normalized = _normalize_sat_params(entry)
            if normalized is not None:
                out[date][str(sat)] = normalized

    return out


def _serialize_params(data):
    payload = {}

    for date in sorted(list_shock_dates()):
        sats = data.get(date, {})
        if not isinstance(sats, dict):
            continue

        sat_payload = {}
        for sat, entry in sats.items():
            normalized = _normalize_sat_params(entry)
            if normalized is None:
                continue

            sat_payload[str(sat)] = {
                "t0": pd.Timestamp(normalized["t0"]).isoformat(),
                "dt0_u": int(normalized["dt0_u"]),
                "dt1_u": int(normalized["dt1_u"]),
                "dt0_d": int(normalized["dt0_d"]),
                "dt1_d": int(normalized["dt1_d"]),
            }

        payload[date] = sat_payload

    return payload


def _save_params(data):
    payload = _serialize_params(data)
    tmp_path = PARAMS_PATH.with_suffix(".json.tmp")
    tmp_path.write_text(json.dumps(payload, indent=2, sort_keys=True))
    tmp_path.replace(PARAMS_PATH)


def _ensure_date_initialized(date):
    """Seed Shock Params.json once for a truly new date."""
    if date in data:
        return data[date]

    seeded = _seed_params_for_date(date, get_dfs(date))
    data[date] = seeded
    if seeded:
        _save_params(data)
    return data[date]


def _get_date_params(date):
    _ensure_date_initialized(date)
    return data.get(date, {})


data = _load_params()

# Provisional (in-memory only) params for satellites not yet confirmed.
# These render in the plot but are NOT saved to JSON until the user moves a slider.
_provisional_params = {}


def _ensure_params_for(date, sat):
    """Ensure in-memory params exist for (date, sat). Does NOT save to JSON."""
    _ensure_date_initialized(date)
    if date in data and sat in data[date] and data[date][sat] is not None:
        return
    if date in _provisional_params and sat in _provisional_params.get(date, {}):
        return
    df = get_dfs(date)[sat]
    t0 = df.index[len(df) // 2]
    _provisional_params.setdefault(date, {})[sat] = {"t0": t0, **DEFAULT_DTS}


def _get_params_for(date, sat):
    """Get params from confirmed data or provisional. Returns None if neither."""
    if date in data and sat in data[date] and data[date][sat] is not None:
        return data[date][sat]
    if date in _provisional_params and sat in _provisional_params.get(date, {}):
        return _provisional_params[date][sat]
    return None


def _is_provisional(date, sat):
    """Check whether (date, sat) params are provisional (not saved to JSON)."""
    return date in _provisional_params and sat in _provisional_params.get(date, {})


def _promote_provisional(date, sat):
    """Move a provisional satellite into confirmed data and save."""
    if not _is_provisional(date, sat):
        return
    entry = _provisional_params[date].pop(sat)
    if not _provisional_params[date]:
        del _provisional_params[date]
    data.setdefault(date, {})[sat] = entry
    _save_params(data)


def _invalidate_sat(date, sat):
    """Remove satellite from confirmed data and provisional. Saves JSON."""
    changed = False
    if date in data and sat in data[date]:
        del data[date][sat]
        changed = True
    if date in _provisional_params and sat in _provisional_params.get(date, {}):
        del _provisional_params[date][sat]
        if not _provisional_params[date]:
            del _provisional_params[date]
    if changed:
        _save_params(data)

## Plotting Function

Main function to visualize shock data with magnetic field, velocity, and density plots. Includes support for multiple coordinate systems and satellite-specific configurations.

In [6]:
def _nearest_index(df, t0):
    # Handle unique index case
    if df.index.is_unique:
        return df.index[df.index.get_indexer([t0], method="nearest")[0]]
    else:
        # For non-unique index, find the closest value manually
        return min(df.index, key=lambda x: abs((x - t0).total_seconds()))


def _plot_window(df, center_t, mode="medium"):
    if mode == "coarse":
        return df

    if mode == "fine":
        delta = pd.Timedelta(seconds=90)
    else:
        delta = pd.Timedelta(minutes=30)

    start = pd.Timestamp(center_t) - delta
    end = pd.Timestamp(center_t) + delta
    df_window = df[start:end]
    return df_window if len(df_window) > 0 else df


def _draw_reference_lines(axis, t0, dt0_u, dt1_u, dt0_d, dt1_d):
    axis.axvline(
        x=pd.Timestamp(t0), color="black", linewidth=2, linestyle="--", label="Shock: " + t0.strftime("%Y-%m-%d %H:%M:%S")
    )
    for key, dt, color in [
        ("dt0_u", dt0_u, "tab:red"),
        ("dt1_u", dt1_u, "tab:orange"),
        ("dt0_d", dt0_d, "tab:blue"),
        ("dt1_d", dt1_d, "tab:purple"),
    ]:
        axis.axvline(
            x=pd.Timestamp(t0) + pd.Timedelta(minutes=dt),
            color=color,
            linewidth=1.2,
            linestyle=":",
            alpha=0.9,
            label="_nolegend_",
        )


def _select_density_column(df_plot):
    for col in ["N_p", "n_p", "Proton_Np_moment", "Np", "N", "proton_density"]:
        if col in df_plot.columns:
            return col
    return None


def plot(
    date, sat, t0=None, dt0_u=None, dt1_u=None, dt0_d=None, dt1_d=None, mode="medium"
):
    dfs = get_dfs(date)
    if sat not in dfs:
        print(f"No data for {sat} in {date}")
        return

    df = dfs[sat]
    _ensure_params_for(date, sat)
    dat = _get_params_for(date, sat)

    t00 = pd.Timestamp(dat["t0"])

    # Ensure timezone consistency
    if len(df.index) > 0:
        if df.index[0].tz is not None and t00.tz is None:
            t00 = t00.tz_localize("UTC")
        elif df.index[0].tz is None and t00.tz is not None:
            t00 = t00.tz_localize(None)

    if t0 is None:
        t0 = t00
    else:
        # Ensure t0 from slider is also timezone-consistent
        t0 = pd.Timestamp(t0)
        if df.index[0].tz is not None and t0.tz is None:
            t0 = t0.tz_localize("UTC")
        elif df.index[0].tz is None and t0.tz is not None:
            t0 = t0.tz_localize(None)
        dat["t0"] = t0

    if dt0_u is None:
        dt0_u = dat["dt0_u"]
    else:
        dat["dt0_u"] = dt0_u
    if dt1_u is None:
        dt1_u = dat["dt1_u"]
    else:
        dat["dt1_u"] = dt1_u
    if dt0_d is None:
        dt0_d = dat["dt0_d"]
    else:
        dat["dt0_d"] = dt0_d
    if dt1_d is None:
        dt1_d = dat["dt1_d"]
    else:
        dat["dt1_d"] = dt1_d

    # Select time range based on the current selected t0
    df_plot = _plot_window(df, t0, mode=mode)

    fig = plt.figure(figsize=(12, 10))
    plt.rcParams["lines.markersize"] = 2.5
    ax = plt.subplot2grid((3, 1), (0, 0), sharex=None)

    marker = "-o" if sat not in high_mag_density else "-"

    ax.set_xlim(df_plot.index[0], df_plot.index[-1])

    def plot_col(a, col, label, color=None):
        if col in df_plot.columns:
            if color is None:
                a.plot(
                    df_plot.dropna(subset=[col]).index,
                    df_plot[col].dropna(),
                    marker,
                    label=label,
                )
            else:
                a.plot(
                    df_plot.dropna(subset=[col]).index,
                    df_plot[col].dropna(),
                    marker,
                    label=label,
                    color=color,
                )

    # Magnetic field - magnitude always
    if "B" in df_plot.columns:
        plot_col(ax, "B", "B")
    elif all(c in df_plot.columns for c in ["B_X_GSE", "B_Y_GSE", "B_Z_GSE"]):
        bmag = np.sqrt(
            df_plot["B_X_GSE"] ** 2 + df_plot["B_Y_GSE"] ** 2 + df_plot["B_Z_GSE"] ** 2
        )
        ax.plot(df_plot.index, bmag, marker, label="B")
    elif all(c in df_plot.columns for c in ["B_X_HGE", "B_Y_HGE", "B_Z_HGE"]):
        bmag = np.sqrt(
            df_plot["B_X_HGE"] ** 2 + df_plot["B_Y_HGE"] ** 2 + df_plot["B_Z_HGE"] ** 2
        )
        ax.plot(df_plot.index, bmag, marker, label="B")
    elif all(c in df_plot.columns for c in ["B_x", "B_y", "B_z"]):
        bmag = np.sqrt(df_plot["B_x"] ** 2 + df_plot["B_y"] ** 2 + df_plot["B_z"] ** 2)
        ax.plot(df_plot.index, bmag, marker, label="B")

    # Plot B components only for non-STEREO (GSE coordinates)
    if sat != "stereo":
        if "B_X_GSE" in df_plot.columns:
            plot_col(ax, "B_X_GSE", "B_x")
            plot_col(ax, "B_Y_GSE", "B_y")
            plot_col(ax, "B_Z_GSE", "B_z")
        else:
            plot_col(ax, "B_x", "B_x")
            plot_col(ax, "B_y", "B_y")
            plot_col(ax, "B_z", "B_z")

    _draw_reference_lines(ax, t0, dt0_u, dt1_u, dt0_d, dt1_d)
    ax.legend(loc="upper left")
    ax.set_ylabel("Magnetic Field (nT)")

    marker = "-o" if sat not in high_v_density else "-"
    plt.rcParams["lines.markersize"] = 5

    ax1 = plt.subplot2grid((3, 1), (1, 0), sharex=ax)
    ax1.set_xlim(df_plot.index[0], df_plot.index[-1])

    # Determine if we're in GSE coordinates
    is_gse = "V_X_GSE" in df_plot.columns or "v_x" in df_plot.columns

    # Velocity plotting with multiple naming convention support
    if "V" in df_plot.columns:
        plot_col(ax1, "V", "V")
    elif "v" in df_plot.columns:
        plot_col(ax1, "v", "V")
    elif "Proton_V_moment" in df_plot.columns:
        plot_col(ax1, "Proton_V_moment", "V")
    elif all(c in df_plot.columns for c in ["V_X_GSE", "V_Y_GSE", "V_Z_GSE"]):
        vmag = np.sqrt(
            df_plot["V_X_GSE"] ** 2 + df_plot["V_Y_GSE"] ** 2 + df_plot["V_Z_GSE"] ** 2
        )
        ax1.plot(df_plot.index, vmag, marker, label="V")
    elif all(c in df_plot.columns for c in ["V_X_HGE", "V_Y_HGE", "V_Z_HGE"]):
        vmag = np.sqrt(
            df_plot["V_X_HGE"] ** 2 + df_plot["V_Y_HGE"] ** 2 + df_plot["V_Z_HGE"] ** 2
        )
        ax1.plot(df_plot.index, vmag, marker, label="V")
    elif all(c in df_plot.columns for c in ["v_x", "v_y", "v_z"]):
        vmag = np.sqrt(df_plot["v_x"] ** 2 + df_plot["v_y"] ** 2 + df_plot["v_z"] ** 2)
        ax1.plot(df_plot.index, vmag, marker, label="V")
    elif all(
        c in df_plot.columns
        for c in ["Proton_VX_moment", "Proton_VY_moment", "Proton_VZ_moment"]
    ):
        vmag = np.sqrt(
            df_plot["Proton_VX_moment"] ** 2
            + df_plot["Proton_VY_moment"] ** 2
            + df_plot["Proton_VZ_moment"] ** 2
        )
        ax1.plot(
            df_plot.dropna(subset=["Proton_VX_moment"]).index,
            vmag[df_plot["Proton_VX_moment"].notna()],
            marker,
            label="V",
        )

    # Plot V_x (or -V_x if GSE) on the first axis
    if "V_X_GSE" in df_plot.columns:
        vx_data = -df_plot["V_X_GSE"] if is_gse else df_plot["V_X_GSE"]
        ax1.plot(
            df_plot.dropna(subset=["V_X_GSE"]).index,
            vx_data.dropna(),
            marker,
            label="-V_x" if is_gse else "V_x",
        )
    elif "v_x" in df_plot.columns:
        vx_data = -df_plot["v_x"] if is_gse else df_plot["v_x"]
        ax1.plot(
            df_plot.dropna(subset=["v_x"]).index,
            vx_data.dropna(),
            marker,
            label="-V_x" if is_gse else "V_x",
        )

    # Create second y-axis for density
    density_col = _select_density_column(df_plot)
    ax1_density = None
    if density_col is not None:
        ax1_density = ax1.twinx()
        ax1_density.plot(
            df_plot.dropna(subset=[density_col]).index,
            df_plot[density_col].dropna(),
            "-",
            color="tab:purple",
            linewidth=1.3,
            alpha=0.85,
            label="N_p",
        )
        ax1_density.set_ylabel("Density (cm^-3)")

    # Create third y-axis for V_y and V_z
    ax1_yz = ax1.twinx()
    ax1_yz.spines["right"].set_position(("outward", 60))

    if "V_Y_GSE" in df_plot.columns and "V_Z_GSE" in df_plot.columns:
        ax1_yz.plot(
            df_plot.dropna(subset=["V_Y_GSE"]).index,
            df_plot["V_Y_GSE"].dropna(),
            marker,
            label="V_y",
            alpha=0.7,
            color="green",
        )
        ax1_yz.plot(
            df_plot.dropna(subset=["V_Z_GSE"]).index,
            df_plot["V_Z_GSE"].dropna(),
            marker,
            label="V_z",
            alpha=0.7,
            color="red",
        )
    elif "v_y" in df_plot.columns and "v_z" in df_plot.columns:
        ax1_yz.plot(
            df_plot.dropna(subset=["v_y"]).index,
            df_plot["v_y"].dropna(),
            marker,
            label="V_y",
            alpha=0.7,
            color="green",
        )
        ax1_yz.plot(
            df_plot.dropna(subset=["v_z"]).index,
            df_plot["v_z"].dropna(),
            marker,
            label="V_z",
            alpha=0.7,
            color="red",
        )
    elif "V_Y_HGE" in df_plot.columns and "V_Z_HGE" in df_plot.columns:
        ax1_yz.plot(
            df_plot.dropna(subset=["V_Y_HGE"]).index,
            df_plot["V_Y_HGE"].dropna(),
            marker,
            label="V_y",
            alpha=0.7,
            color="green",
        )
        ax1_yz.plot(
            df_plot.dropna(subset=["V_Z_HGE"]).index,
            df_plot["V_Z_HGE"].dropna(),
            marker,
            label="V_z",
            alpha=0.7,
            color="red",
        )

    ax1_yz.set_ylabel("V_y, V_z (km/s)")

    _draw_reference_lines(ax1, t0, dt0_u, dt1_u, dt0_d, dt1_d)

    # Combine all legends
    handles, labels = ax1.get_legend_handles_labels()
    if ax1_density is not None:
        h2, l2 = ax1_density.get_legend_handles_labels()
        handles.extend(h2)
        labels.extend(l2)
    h3, l3 = ax1_yz.get_legend_handles_labels()
    handles.extend(h3)
    labels.extend(l3)
    ax1.legend(handles, labels, loc="upper left")
    ax1.set_ylabel("V, V_x (km/s)")

    fig.tight_layout()
    plt.show()
    if not _is_provisional(date, sat):
        _save_params(data)

## Interactive Controls

Create widgets for interactive parameter tuning: date selection, satellite selection, shock time (t0), and upstream/downstream time deltas.

In [7]:
dates = list_shock_dates()
if not dates:
    print("No shock folders found in Data")

date_dropdown = widgets.Dropdown(
    options=dates, description="Date", layout=widgets.Layout(width="50%")
)
sat_dropdown = widgets.Dropdown(
    options=[pd.Timestamp("1970-01-01")],
    description="Sat",
    layout=widgets.Layout(width="50%"),
)

t0_slider = widgets.SelectionSlider(
    options=[pd.Timestamp("1970-01-01")],
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    layout=widgets.Layout(width="75%"),
)

dt0_u_slider = widgets.IntSlider(
    value=DEFAULT_DTS["dt0_u"],
    max=-1,
    min=-60,
    step=1,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
    description="dt0_u (min)",
    layout=widgets.Layout(width="75%"),
)
dt1_u_slider = widgets.IntSlider(
    value=DEFAULT_DTS["dt1_u"],
    max=-1,
    min=-60,
    step=1,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
    description="dt1_u (min)",
    layout=widgets.Layout(width="75%"),
)
dt0_d_slider = widgets.IntSlider(
    value=DEFAULT_DTS["dt0_d"],
    min=1,
    max=60,
    step=1,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
    description="dt0_d (min)",
    layout=widgets.Layout(width="75%"),
)
dt1_d_slider = widgets.IntSlider(
    value=DEFAULT_DTS["dt1_d"],
    min=1,
    max=60,
    step=1,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
    description="dt1_d (min)",
    layout=widgets.Layout(width="75%"),
)

mode_radio = widgets.RadioButtons(
    options=[("Coarse", "coarse"), ("Medium", "medium"), ("Fine", "fine")],
    value="medium",
    description="Mode",
    layout=widgets.Layout(width="50%"),
)

invalidate_btn = widgets.Button(
    description="Invalidate",
    button_style="danger",
    tooltip="Remove this satellite from Shock Params.json",
    layout=widgets.Layout(width="auto"),
)

status_label = widgets.Label(value="")

_control_update_in_progress = False
_last_sat = None


def _update_status_label():
    date = date_dropdown.value
    sat = sat_dropdown.value
    if not date or not sat:
        status_label.value = ""
        return
    if _is_provisional(date, sat):
        status_label.value = "\u26a0 Unsaved (move a slider to confirm)"
    elif date in data and sat in data.get(date, {}):
        status_label.value = "\u2705 Saved"
    else:
        status_label.value = ""


def _render(*_):
    if _control_update_in_progress:
        return
    if not date_dropdown.value or not sat_dropdown.value:
        return
    with out:
        clear_output(wait=True)
        plot(
            date=date_dropdown.value,
            sat=sat_dropdown.value,
            t0=t0_slider.value,
            dt0_u=dt0_u_slider.value,
            dt1_u=dt1_u_slider.value,
            dt0_d=dt0_d_slider.value,
            dt1_d=dt1_d_slider.value,
            mode=mode_radio.value,
        )
    _update_status_label()


def _on_slider_change(*_):
    """Promote provisional params on any slider interaction, then render."""
    if _control_update_in_progress:
        return
    date = date_dropdown.value
    sat = sat_dropdown.value
    if date and sat and _is_provisional(date, sat):
        _promote_provisional(date, sat)
    _render()


def _on_invalidate(*_):
    date = date_dropdown.value
    sat = sat_dropdown.value
    if not date or not sat:
        return
    _invalidate_sat(date, sat)
    # Re-create provisional entry so the plot still renders
    _ensure_params_for(date, sat)
    _update_status_label()
    _render()


def _refresh_sats(*_):
    if not date_dropdown.value:
        sat_dropdown.options = []
        return
    sats = sorted(get_dfs(date_dropdown.value).keys())
    sat_dropdown.options = sats
    if sats and sat_dropdown.value not in sats:
        sat_dropdown.value = sats[0]
    else:
        _refresh_controls()


def _nearest_from_index(idx, t0):
    if idx.is_unique:
        return idx[idx.get_indexer([t0], method="nearest")[0]]
    return min(idx, key=lambda x: abs((x - t0).total_seconds()))


def _refresh_controls(*_):
    global _control_update_in_progress, _last_sat
    if _control_update_in_progress:
        return
    if not date_dropdown.value or not sat_dropdown.value:
        return

    _control_update_in_progress = True
    try:
        date = date_dropdown.value
        sat = sat_dropdown.value
        dfs = get_dfs(date)
        df = dfs[sat]
        _ensure_params_for(date, sat)
        dat = _get_params_for(date, sat)

        sat_changed = sat != _last_sat
        _last_sat = sat

        t0 = pd.Timestamp(dat["t0"])

        if len(df.index) > 0:
            idx_start = df.index[0]
            idx_end = df.index[-1]
            if idx_start.tz is not None and t0.tz is None:
                t0 = t0.tz_localize("UTC")
            elif idx_start.tz is None and t0.tz is not None:
                t0 = t0.tz_localize(None)

            # Only carry over the current slider value when the satellite
            # has NOT changed (e.g. mode switch).  When the satellite
            # changes we must use the value stored in JSON to avoid
            # cross-pollination from the previous satellite.
            if not sat_changed and t0_slider.value is not None:
                current_t0 = pd.Timestamp(t0_slider.value)
                if idx_start.tz is not None and current_t0.tz is None:
                    current_t0 = current_t0.tz_localize("UTC")
                elif idx_start.tz is None and current_t0.tz is not None:
                    current_t0 = current_t0.tz_localize(None)
                if idx_start <= current_t0 <= idx_end:
                    t0 = current_t0

        if t0 < df.index[0] or t0 > df.index[-1]:
            t0 = df.index[len(df) // 2]
            dat["t0"] = t0

        mode = mode_radio.value
        if mode == "coarse":
            t0_options = df.index
        elif mode == "fine":
            t_start = t0 - pd.Timedelta(seconds=90)
            t_end = t0 + pd.Timedelta(seconds=90)
            t0_options = df[t_start:t_end].index
            if len(t0_options) == 0:
                t0_options = df.index
        else:
            t_start = t0 - pd.Timedelta(minutes=30)
            t_end = t0 + pd.Timedelta(minutes=30)
            t0_options = df[t_start:t_end].index
            if len(t0_options) == 0:
                t0_options = df.index

        if tuple(t0_slider.options) != tuple(t0_options):
            t0_slider.options = t0_options

        nearest_t0 = _nearest_from_index(t0_options, t0)
        if t0_slider.value != nearest_t0:
            t0_slider.value = nearest_t0

        new_dt0_u = dat.get("dt0_u", DEFAULT_DTS["dt0_u"])
        new_dt1_u = dat.get("dt1_u", DEFAULT_DTS["dt1_u"])
        new_dt0_d = dat.get("dt0_d", DEFAULT_DTS["dt0_d"])
        new_dt1_d = dat.get("dt1_d", DEFAULT_DTS["dt1_d"])

        if dt0_u_slider.value != new_dt0_u:
            dt0_u_slider.value = new_dt0_u
        if dt1_u_slider.value != new_dt1_u:
            dt1_u_slider.value = new_dt1_u
        if dt0_d_slider.value != new_dt0_d:
            dt0_d_slider.value = new_dt0_d
        if dt1_d_slider.value != new_dt1_d:
            dt1_d_slider.value = new_dt1_d
    finally:
        _control_update_in_progress = False

    _update_status_label()
    _render()


out = widgets.Output()

# Refresh chain observers
# Date only refreshes satellites; controls refresh is driven from sat selection.
date_dropdown.observe(_refresh_sats, names="value")
sat_dropdown.observe(_refresh_controls, names="value")
mode_radio.observe(_refresh_controls, names="value")

# Slider observers — promote provisional on any slider interaction
t0_slider.observe(_on_slider_change, names="value")
dt0_u_slider.observe(_on_slider_change, names="value")
dt1_u_slider.observe(_on_slider_change, names="value")
dt0_d_slider.observe(_on_slider_change, names="value")
dt1_d_slider.observe(_on_slider_change, names="value")

invalidate_btn.on_click(_on_invalidate)

_refresh_sats()
_refresh_controls()

controls = widgets.VBox(
    [
        widgets.HBox([date_dropdown, sat_dropdown]),
        mode_radio,
        t0_slider,
        dt0_u_slider,
        dt1_u_slider,
        dt0_d_slider,
        dt1_d_slider,
        widgets.HBox([invalidate_btn, status_label]),
    ]
)

## Display

Show the interactive controls and output area for plots.

In [8]:
display(controls, out)

Output()

## Debugging / Testing

Utility cells for checking data availability and column structure.

In [9]:
# Check STEREO columns if available
for date in dates:
    dfs = get_dfs(date)
    if "stereo" in dfs:
        df = dfs["stereo"]
        print(f"STEREO data available for {date}")
        print(f"Columns: {df.columns.tolist()}")
        print(f"B columns: {[c for c in df.columns if 'B' in c.upper()]}")
        print(f"V columns: {[c for c in df.columns if 'v' in c.lower()]}")
        break
else:
    print("No STEREO data found in available dates")

STEREO data available for 2022-01-16 19-09-00
Columns: ['B_X_HGE', 'B_Y_HGE', 'B_Z_HGE', 'B', 'N_p', 'V', 'T_p', 'radialDistance', 'heliographicLatitude', 'heliographicLongitude']
B columns: ['B_X_HGE', 'B_Y_HGE', 'B_Z_HGE', 'B']
V columns: ['V']


In [10]:
# Example: Plot normals for the current date and satellite
# Uncomment and run after selecting a date/satellite above

# if date_dropdown.value and sat_dropdown.value:
#     plot_normals_polar(date_dropdown.value, sat_dropdown.value)
#     plot_normals_2d_planes(date_dropdown.value)

In [11]:
def plot_normals_2d_planes(date, satellites=None, method_index=3):
    """
    Plot shock normals in 2D planes (X-Y and X-Z).
    Shows satellite positions and shock normal tangent lines.

    Parameters:
    - date: shock date
    - satellites: list of satellites to include (default: all available)
    - method_index: which method to use (0=MC, 1=MX1, 2=MX2, 3=MX3, 4=VC)
    """
    dfs_dict = get_dfs(date)

    if satellites is None:
        satellites = list(dfs_dict.keys())

    # Satellite colors
    sat_colors = {
        "ace": "#ce4a4a",
        "wind": "#6688c3",
        "dscovr": "#48a56a",
        "themis": "#b25da6",
        "stereo": "#f4a460",
    }

    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(14, 7))

    # Placeholder satellite positions (will be replaced with actual positions from JSON)
    # For now, using simple spacing
    sat_positions = {}
    for i, sat in enumerate(satellites):
        sat_positions[sat] = {"x": 150 + i * 20, "y": -50 + i * 30, "z": -20 + i * 25}

    # Plot X-Y plane
    for sat in satellites:
        if sat not in dfs_dict:
            continue

        pos = sat_positions.get(sat, {"x": 0, "y": 0, "z": 0})
        color = sat_colors.get(sat, "#888888")

        # Plot satellite position
        ax[0].scatter(
            pos["x"], pos["y"], marker="o", s=100, color=color, label=sat.upper()
        )

        # Compute and plot normal line
        thetas, phis = compute_normals(date, sat)
        if thetas is not None and not np.isnan(thetas[method_index]):
            angle = thetas[method_index]
            # Draw line perpendicular to normal (shock tangent)
            x1 = pos["x"] + 50 * np.cos(np.radians(angle))
            y1 = pos["y"] + 50 * np.sin(np.radians(angle))
            x2 = pos["x"] - 50 * np.cos(np.radians(angle))
            y2 = pos["y"] - 50 * np.sin(np.radians(angle))
            ax[0].plot([x1, x2], [y1, y2], "-", color="#b25da6", linewidth=2, alpha=0.7)

    # Plot X-Z plane
    for sat in satellites:
        if sat not in dfs_dict:
            continue

        pos = sat_positions.get(sat, {"x": 0, "y": 0, "z": 0})
        color = sat_colors.get(sat, "#888888")

        # Plot satellite position
        ax[1].scatter(pos["x"], pos["z"], marker="o", s=100, color=color)

        # Compute and plot normal line
        thetas, phis = compute_normals(date, sat)
        if phis is not None and not np.isnan(phis[method_index]):
            angle = phis[method_index]
            # Draw line perpendicular to normal (shock tangent)
            x1 = pos["x"] + 50 * np.cos(np.radians(angle))
            z1 = pos["z"] + 50 * np.sin(np.radians(angle))
            x2 = pos["x"] - 50 * np.cos(np.radians(angle))
            z2 = pos["z"] - 50 * np.sin(np.radians(angle))
            ax[1].plot([x1, x2], [z1, z2], "-", color="#b25da6", linewidth=2, alpha=0.7)

    # Add Earth
    for a in ax:
        a.add_patch(plt.Circle((0, 0), 60, color="lightgray", zorder=0))
        a.add_patch(plt.Circle((0, 0), 53, color="white", zorder=1))
        a.axhline(y=0, color="black", linewidth=0.5, alpha=0.5)
        a.axvline(x=0, color="black", linewidth=0.5, alpha=0.5)
        a.set_xlim(-25, 300)
        a.set_ylim(-200, 200)
        a.set_aspect("equal")

    ax[0].set_title("X-Y Plane", fontsize=14)
    ax[0].set_xlabel("X (GSE)", fontsize=12)
    ax[0].set_ylabel("Y (GSE)", fontsize=12)
    ax[0].invert_xaxis()
    ax[0].invert_yaxis()

    ax[1].set_title("X-Z Plane", fontsize=14)
    ax[1].set_xlabel("X (GSE)", fontsize=12)
    ax[1].set_ylabel("Z (GSE)", fontsize=12)
    ax[1].invert_xaxis()

    method_names = ["MC", "MX1", "MX2", "MX3", "VC"]
    fig.suptitle(
        f"Shock Normals - {date} (Method: {method_names[method_index]})", fontsize=16
    )

    # Create legend
    handles = [
        mpatches.Patch(color=sat_colors.get(s, "#888"), label=s.upper())
        for s in satellites
        if s in dfs_dict
    ]
    handles.append(mpatches.Patch(color="#b25da6", label="Shock Tangent"))
    ax[0].legend(handles=handles, loc="upper left")

    fig.tight_layout()
    plt.show()

In [12]:
def plot_normals_polar(date, sat, offset_u=-5, offset_d=5, interval=5):
    """
    Plot shock normals in 2D polar plots for theta and phi angles.
    Each method (MC, MX1, MX2, MX3, VC) is shown as a point.
    """
    thetas, phis = compute_normals(date, sat, offset_u, offset_d, interval)

    if thetas is None or phis is None:
        return

    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, ncols=2, figsize=(12, 5))

    colors = ["#ce4a4a", "#6688c3", "#48a56a", "#b25da6", "#f4a460"]
    labels = ["MC", "MX1", "MX2", "MX3", "VC"]

    for i, (theta, phi) in enumerate(zip(thetas, phis)):
        if not np.isnan(theta):
            ax[0].scatter(np.radians(theta), 1, s=100, color=colors[i], label=labels[i])
        if not np.isnan(phi):
            ax[1].scatter(np.radians(phi), 1, s=100, color=colors[i], label=labels[i])

    ax[0].set_title(r"$\theta$ (degrees)", fontsize=14)
    ax[0].set_ylim(0, 1.2)
    ax[0].set_yticklabels([])

    ax[1].set_title(r"$\phi$ (degrees)", fontsize=14)
    ax[1].set_ylim(0, 1.2)
    ax[1].set_yticklabels([])

    fig.suptitle(f"{sat.upper()} - {date}", fontsize=16)
    ax[0].legend(labels, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 0.95))
    fig.tight_layout()
    plt.show()

In [13]:
def compute_normals(date, sat, offset_u=-5, offset_d=5, interval=5):
    """
    Compute shock normals using all MHD methods.
    Returns theta and phi angles in degrees for each method.
    """
    dfs = get_dfs(date)
    if sat not in dfs:
        print(f"No data for {sat} in {date}")
        return None, None

    df = dfs[sat]
    _ensure_params_for(date, sat)
    dat = data[date][sat]
    event_time = dat["t0"]

    # Extract velocity and magnetic field
    v1, v2 = _extract_velocity(df, event_time, offset_u, offset_d, interval)
    b1, b2 = _extract_bfield(df, event_time, offset_u, offset_d, interval)

    if v1 is None or b1 is None or any(np.isnan(v1)) or any(np.isnan(b1)):
        print(f"Insufficient data for {sat}")
        return None, None

    delta_v = np.array(v2) - np.array(v1)
    b1_arr = np.array(b1)
    b2_arr = np.array(b2)

    thetas = []
    phis = []

    # Compute normals using all methods
    methods = [mc, mx1, mx2, mx3, vc]
    for method in methods:
        try:
            normal = method(b1_arr, b2_arr, delta_v)
            r, theta, phi = astropy.coordinates.cartesian_to_spherical(*normal)
            thetas.append(theta.degree)
            phis.append(phi.degree)
        except Exception as e:
            print(f"Error in {method.__name__}: {e}")
            thetas.append(np.nan)
            phis.append(np.nan)

    return thetas, phis

In [14]:
# Helper functions for normal calculation


def _get_time_slice(df, event_time, dt1, dt2):
    """Extract time slice from dataframe"""
    return df[
        pd.Timestamp(event_time)
        + pd.Timedelta(minutes=dt1) : pd.Timestamp(event_time)
        + pd.Timedelta(minutes=dt2)
    ]


def _extract_velocity(df, event_time, offset_u=-5, offset_d=5, interval=5):
    """Extract upstream and downstream velocity vectors"""
    # Determine column names based on coordinate system
    if "v_x" in df.columns or "V_X_GSE" in df.columns:
        v_cols = (
            ["v_x", "v_y", "v_z"]
            if "v_x" in df.columns
            else ["V_X_GSE", "V_Y_GSE", "V_Z_GSE"]
        )
    elif "v_r" in df.columns:
        v_cols = ["v_r", "v_t", "v_n"]
    elif "V_X_HGE" in df.columns:
        v_cols = ["V_X_HGE", "V_Y_HGE", "V_Z_HGE"]
    else:
        return None, None

    v1 = [
        _get_time_slice(df, event_time, offset_u - interval, offset_u)[col].mean()
        for col in v_cols
    ]
    v2 = [
        _get_time_slice(df, event_time, offset_d, offset_d + interval)[col].mean()
        for col in v_cols
    ]
    return v1, v2


def _extract_bfield(df, event_time, offset_u=-5, offset_d=5, interval=5):
    """Extract upstream and downstream magnetic field vectors"""
    # Determine column names based on coordinate system
    if "B_x" in df.columns or "B_X_GSE" in df.columns:
        b_cols = (
            ["B_x", "B_y", "B_z"]
            if "B_x" in df.columns
            else ["B_X_GSE", "B_Y_GSE", "B_Z_GSE"]
        )
    elif "B_r" in df.columns:
        b_cols = ["B_r", "B_t", "B_n"]
    elif "B_X_HGE" in df.columns:
        b_cols = ["B_X_HGE", "B_Y_HGE", "B_Z_HGE"]
    else:
        return None, None

    b1 = [
        _get_time_slice(df, event_time, offset_u - interval, offset_u)[col].mean()
        for col in b_cols
    ]
    b2 = [
        _get_time_slice(df, event_time, offset_d, offset_d + interval)[col].mean()
        for col in b_cols
    ]
    return b1, b2

In [15]:
# MHD Normal Calculation Methods


def mc(b1, b2, delta_v):
    """Magnetic coplanarity method"""
    delta_b = np.array(b2) - np.array(b1)
    m = np.cross(np.cross(b2, b1), delta_b)
    n = m / np.linalg.norm(m)
    return n


def mx1(b1, b2, delta_v):
    """Mixed method 1: uses upstream B and velocity"""
    delta_b = np.array(b2) - np.array(b1)
    m = np.cross(np.cross(b1, delta_v), delta_b)
    n = m / np.linalg.norm(m)
    return n


def mx2(b1, b2, delta_v):
    """Mixed method 2: uses downstream B and velocity"""
    delta_b = np.array(b2) - np.array(b1)
    m = np.cross(np.cross(b2, delta_v), delta_b)
    n = m / np.linalg.norm(m)
    return n


def mx3(b1, b2, delta_v):
    """Mixed method 3: uses delta B and velocity"""
    delta_b = np.array(b2) - np.array(b1)
    m = np.cross(np.cross(delta_b, delta_v), delta_b)
    n = m / np.linalg.norm(m)
    return n


def vc(b1, b2, delta_v):
    """Velocity coplanarity method"""
    m = delta_v
    n = m / np.linalg.norm(m)
    return n

In [16]:
# Load shock normals from Wolfram JSON file
import json
from pathlib import Path

NORMALS_JSON_PATH = Path("Shocks/Kinematic Shock Normals.json")


def load_shock_normals():
    """Load sphere and planar normals from JSON file"""
    if not NORMALS_JSON_PATH.exists():
        print(f"Normals file not found: {NORMALS_JSON_PATH}")
        return None

    with open(NORMALS_JSON_PATH, "r") as f:
        return json.load(f)


shock_normals_data = load_shock_normals()


def get_normals_for_date(date_str):
    """Get planar and spherical normals for a specific date"""
    if shock_normals_data is None:
        return None, None

    # Format date string to match JSON keys
    date_key = None
    for key in shock_normals_data.keys():
        # Try to match various date formats
        if date_str.replace(" ", "").replace("-", "").replace(":", "") in key.replace(
            " ", ""
        ).replace("-", "").replace(":", ""):
            date_key = key
            break

    if date_key is None:
        return None, None

    entry = shock_normals_data.get(date_key, {})
    planar = entry.get("planar")
    spherical = entry.get("spherical")

    return planar, spherical

In [17]:
from matplotlib.lines import Line2D

R_E = 6371  # Earth radius in km


def theta_phi(n):
    """Convert normal vector to theta (X-Z) and phi (Y-Z) angles.
    Convention from Storm Analysis.ipynb:
      theta = pi/2 + angle(n, OZ)
      phi   = pi - arcsin(n_y / sqrt(n_x^2 + n_y^2))
    """
    theta = np.degrees(np.pi / 2 + np.arccos(np.dot(n, [0, 0, 1]) / np.linalg.norm(n)))
    phi = np.degrees(np.pi - np.arcsin(n[1] / np.sqrt(n[0] ** 2 + n[1] ** 2)))
    return theta, phi


def front_angle(a):
    """Shift angle into the front frame (+90, wrapped to [0, 360))."""
    b = a + 90
    if b < 0:
        b += 360
    elif b >= 360:
        b -= 360
    return b


_SATS_ALREADY_RE = {"wind"}  # Wind parquets store positions in R_E


def _get_sat_params(params, sat_name):
    sat = str(sat_name).lower()
    candidates = [sat]
    if sat == "mms1":
        candidates.append("mms_1")
    if sat == "mms_1":
        candidates.append("mms1")
    for key in candidates:
        if key in params and params[key] is not None:
            return params[key]
    return None


def _pick_position_columns(df):
    for cols in [
        ("X_GSE", "Y_GSE", "Z_GSE"),
        ("x_gse", "y_gse", "z_gse"),
        ("X", "Y", "Z"),
        ("x", "y", "z"),
    ]:
        if all(c in df.columns for c in cols):
            return cols
    return None


def _coords_already_re(x, y, z, sat_name=None):
    sat = str(sat_name).lower() if sat_name is not None else ""
    if sat in _SATS_ALREADY_RE:
        return True
    max_abs = np.nanmax(np.abs([x, y, z]))
    return max_abs < 1000


def get_sat_position_re(df, t0, sat_name=None):
    """Return satellite GSE position in R_E at the time nearest to *t0*.
    If nearby coordinates are NaN, falls back to the nearest valid coordinate row.
    """
    pos_cols = _pick_position_columns(df)
    if pos_cols is None:
        return None

    pos_df = df.loc[:, list(pos_cols)].copy()
    pos_df = pos_df.apply(pd.to_numeric, errors="coerce")
    pos_df = pos_df.sort_index()
    pos_df = pos_df[~pos_df.index.duplicated(keep="first")]

    t0_ts = pd.Timestamp(t0)
    if pos_df.index.tz is not None and t0_ts.tz is None:
        t0_ts = t0_ts.tz_localize("UTC")
    elif pos_df.index.tz is None and t0_ts.tz is not None:
        t0_ts = t0_ts.tz_localize(None)

    idx = pos_df.index.get_indexer([t0_ts], method="nearest")[0]
    row = pos_df.iloc[idx]
    x, y, z = row[pos_cols[0]], row[pos_cols[1]], row[pos_cols[2]]

    if any(np.isnan(v) for v in (x, y, z)):
        window = pos_df[
            t0_ts - pd.Timedelta(minutes=60) : t0_ts + pd.Timedelta(minutes=60)
        ]
        valid = window.dropna(subset=list(pos_cols))
        if valid.empty:
            valid = pos_df.dropna(subset=list(pos_cols))
            if valid.empty:
                return None
        nearest_idx = valid.index.get_indexer([t0_ts], method="nearest")[0]
        row = valid.iloc[nearest_idx]
        x, y, z = row[pos_cols[0]], row[pos_cols[1]], row[pos_cols[2]]

    if not _coords_already_re(x, y, z, sat_name=sat_name):
        x, y, z = x / R_E, y / R_E, z / R_E

    return {"x": float(x), "y": float(y), "z": float(z)}


def compute_mx3_normal(df, t0, dt0_u, dt1_u, dt0_d, dt1_d):
    """Compute MX3 shock normal from B-field and velocity data."""
    try:
        # Handle tz-mismatch: make t0 tz-aware if index is tz-aware
        t0_ts = pd.Timestamp(t0)
        if hasattr(df.index, "tz") and df.index.tz is not None and t0_ts.tz is None:
            t0_ts = t0_ts.tz_localize("UTC")

        b1, b2 = _extract_bfield(
            df, t0_ts, offset_u=dt1_u, offset_d=dt0_d, interval=abs(dt1_u - dt0_u)
        )
        v1, v2 = _extract_velocity(
            df, t0_ts, offset_u=dt1_u, offset_d=dt0_d, interval=abs(dt1_d - dt0_d)
        )
        if b1 is None or v1 is None:
            return None
        if any(np.isnan(b1)) or any(np.isnan(b2)):
            return None
        if any(np.isnan(v1)) or any(np.isnan(v2)):
            return None
        delta_v = np.array(v2) - np.array(v1)
        if np.linalg.norm(delta_v) < 1e-10:
            return None
        n = mx3(b1, b2, delta_v)
        if np.any(np.isnan(n)):
            return None
        return n
    except Exception:
        return None


def compute_mc_normal(df, t0, dt0_u, dt1_u, dt0_d, dt1_d):
    """Compute magnetic coplanarity (MC) shock normal from B-field data."""
    t0_ts = pd.Timestamp(t0)
    if hasattr(df.index, "tz") and df.index.tz is not None and t0_ts.tz is None:
        t0_ts = t0_ts.tz_localize("UTC")

    b1, b2 = _extract_bfield(
        df,
        t0_ts,
        offset_u=dt1_u,
        offset_d=dt0_d,
        interval=abs(dt1_u - dt0_u),
    )
    if b1 is None:
        return None
    if any(np.isnan(b1)) or any(np.isnan(b2)):
        return None

    n = mc(b1, b2, np.array([0.0, 0.0, 0.0]))
    if not np.isfinite(n).all():
        return None
    return n


def _pick_l1_center(dfs_local, params):
    """Pick sphere center from available L1 satellites (ACE/Wind/DSCOVR)."""
    for sat in ("ace", "wind", "dscovr"):
        if sat not in dfs_local:
            continue
        p = _get_sat_params(params, sat)
        if p is None:
            continue
        pos = get_sat_position_re(dfs_local[sat], p["t0"], sat_name=sat)
        if pos is not None:
            return float(pos["x"]), float(pos["y"]), float(pos["z"]), sat
    return None, None, None, None


def plot_shock_normals(date):
    """
    SA2-style 2-panel plot: (-X)(-Y) left and (-X)Z right.
    Shows MX3 (dashed) and MC (dotted) normals for each satellite,
    planar normal (gray dash-dot), and the fitted sphere projection.
    """
    dfs_local = get_dfs(date)
    params = _get_date_params(date)
    planar_data, spherical_data = get_normals_for_date(date)

    fig, (ax_xy, ax_xz) = plt.subplots(
        ncols=2, figsize=(8, 5), gridspec_kw={"wspace": 0.4}
    )

    for axi in (ax_xy, ax_xz):
        axi.tick_params(
            axis="both",
            which="both",
            top=True,
            bottom=True,
            left=True,
            right=True,
            direction="in",
        )
        for spine in ["top", "bottom", "left", "right"]:
            axi.spines[spine].set_visible(True)

    ax_xy.set_xlim(-150, 400)
    ax_xy.set_ylim(-200, 300)
    ax_xz.set_xlim(-150, 400)
    ax_xz.set_ylim(-200, 300)

    for ax in (ax_xy, ax_xz):
        ax.axhline(y=0, color="black", linewidth=0.5)
        ax.axvline(x=0, color="black", linewidth=0.5)

    l1_x, l1_y, l1_z, l1_sat = _pick_l1_center(dfs_local, params)

    if spherical_data is not None and spherical_data.get("status") == "ok":
        x_fit, y_fit, z_fit = spherical_data["center"]
        r0 = spherical_data["r0"]
        x0, y0, z0 = x_fit, y_fit, z_fit

        if r0**2 > z0**2:
            r_xy = np.sqrt(r0**2 - z0**2)
            ax_xy.add_patch(plt.Circle((x0, y0), r_xy, color="#e5e5fd", zorder=0))

        if r0**2 > y0**2:
            r_xz = np.sqrt(r0**2 - y0**2)
            ax_xz.add_patch(plt.Circle((x0, z0), r_xz, color="#e5e5fd", zorder=0))

    excluded_sats = {"stereo", "themis_c"}
    sat_list = []
    sat_params = {}
    for sat0 in dfs_local:
        sat_key = str(sat0).lower()
        p = _get_sat_params(params, sat_key)
        if p is None:
            continue
        if sat_key in excluded_sats or sat_key.startswith("stereo"):
            continue
        sat_list.append(sat0)
        sat_params[sat0] = p

    line_len = 50
    sat_colors = {sat0: f"C{i}" for i, sat0 in enumerate(sat_list)}

    for sat0 in sat_list:
        p = sat_params[sat0]
        pos = get_sat_position_re(dfs_local[sat0], p["t0"], sat_name=sat0)
        if pos is None:
            continue

        color = sat_colors[sat0]
        ax_xy.scatter(pos["x"], pos["y"], label=sat0, color=color, zorder=999)

        n_mx3 = compute_mx3_normal(
            dfs_local[sat0], p["t0"], p["dt0_u"], p["dt1_u"], p["dt0_d"], p["dt1_d"]
        )
        if n_mx3 is not None:
            _, phi_m = theta_phi(n_mx3)
            phi_m_front = front_angle(phi_m)
            x = pos["x"]
            y = pos["y"]
            endy = y + line_len * np.sin(np.radians(phi_m_front))
            endx = x + line_len * np.cos(np.radians(phi_m_front))
            starty = y - line_len * np.sin(np.radians(phi_m_front))
            startx = x - line_len * np.cos(np.radians(phi_m_front))
            ax_xy.plot([startx, endx], [starty, endy], "--", color=color)

        n_mc = compute_mc_normal(
            dfs_local[sat0], p["t0"], p["dt0_u"], p["dt1_u"], p["dt0_d"], p["dt1_d"]
        )
        if n_mc is not None:
            _, phi_c = theta_phi(n_mc)
            phi_c_front = front_angle(phi_c)
            x = pos["x"]
            y = pos["y"]
            endy = y + line_len * np.sin(np.radians(phi_c_front))
            endx = x + line_len * np.cos(np.radians(phi_c_front))
            starty = y - line_len * np.sin(np.radians(phi_c_front))
            startx = x - line_len * np.cos(np.radians(phi_c_front))
            ax_xy.plot([startx, endx], [starty, endy], ":", color=color)

    if planar_data is not None and planar_data.get("status") == "ok":
        n_planar = np.array(planar_data["normal"])
        _, phi_p = theta_phi(n_planar)
        phi_p_front = front_angle(phi_p)
        if l1_x is not None and l1_y is not None:
            x_ref, y_ref = l1_x, l1_y
        else:
            x_ref, y_ref = 150, 0
        endy = y_ref + 150 * np.sin(np.radians(phi_p_front))
        endx = x_ref + 150 * np.cos(np.radians(phi_p_front))
        starty = y_ref - 150 * np.sin(np.radians(phi_p_front))
        startx = x_ref - 150 * np.cos(np.radians(phi_p_front))
        ax_xy.plot([startx, endx], [starty, endy], "-.", color="gray")

    for sat0 in sat_list:
        p = sat_params[sat0]
        pos = get_sat_position_re(dfs_local[sat0], p["t0"], sat_name=sat0)
        if pos is None:
            continue

        color = sat_colors[sat0]
        ax_xz.scatter(pos["x"], pos["z"], label=sat0, color=color, zorder=999)

        n_mx3 = compute_mx3_normal(
            dfs_local[sat0], p["t0"], p["dt0_u"], p["dt1_u"], p["dt0_d"], p["dt1_d"]
        )
        if n_mx3 is not None:
            theta_m, _ = theta_phi(n_mx3)
            theta_m_front = front_angle(theta_m)
            x = pos["x"]
            z = pos["z"]
            endz = z + line_len * np.sin(np.radians(theta_m_front))
            endx = x + line_len * np.cos(np.radians(theta_m_front))
            startz = z - line_len * np.sin(np.radians(theta_m_front))
            startx = x - line_len * np.cos(np.radians(theta_m_front))
            ax_xz.plot([startx, endx], [startz, endz], "--", color=color)

        n_mc = compute_mc_normal(
            dfs_local[sat0], p["t0"], p["dt0_u"], p["dt1_u"], p["dt0_d"], p["dt1_d"]
        )
        if n_mc is not None:
            theta_c, _ = theta_phi(n_mc)
            theta_c_front = front_angle(theta_c)
            x = pos["x"]
            z = pos["z"]
            endz = z + line_len * np.sin(np.radians(theta_c_front))
            endx = x + line_len * np.cos(np.radians(theta_c_front))
            startz = z - line_len * np.sin(np.radians(theta_c_front))
            startx = x - line_len * np.cos(np.radians(theta_c_front))
            ax_xz.plot([startx, endx], [startz, endz], ":", color=color)

    if planar_data is not None and planar_data.get("status") == "ok":
        n_planar = np.array(planar_data["normal"])
        theta_p, _ = theta_phi(n_planar)
        theta_p_front = front_angle(theta_p)
        if l1_x is not None and l1_z is not None:
            x_ref, z_ref = l1_x, l1_z
        else:
            x_ref, z_ref = 150, 0
        endz = z_ref + 100 * np.sin(np.radians(theta_p_front))
        endx = x_ref + 100 * np.cos(np.radians(theta_p_front))
        startz = z_ref - 100 * np.sin(np.radians(theta_p_front))
        startx = x_ref - 100 * np.cos(np.radians(theta_p_front))
        ax_xz.plot([startx, endx], [startz, endz], "-.", color="gray")

    ax_xy.set_aspect("equal")
    ax_xz.set_aspect("equal")

    ax_xy.set_xlabel("-X (GSE), $R_E$")
    ax_xy.set_ylabel("-Y (GSE), $R_E$")
    ax_xz.set_xlabel("-X (GSE), $R_E$")
    ax_xz.set_ylabel("Z (GSE), $R_E$")

    ax_xy.invert_xaxis()
    ax_xy.invert_yaxis()
    ax_xz.invert_xaxis()

    legend1 = ax_xy.legend(loc="lower right", fontsize=8)
    ax_xy.legend(
        loc="lower left",
        handles=[
            Line2D([0], [0], ls="--", label="MX3", color="k"),
            Line2D([0], [0], ls=":", label="MC", color="k"),
            matplotlib.patches.Rectangle(
                (0, 0), 1, 1, facecolor="#e5e5fd", label="Sphere"
            ),
            Line2D([0], [0], ls="-.", label="Surface", color="gray"),
        ],
        fontsize=8
    )
    ax_xy.add_artist(legend1)

    fig.suptitle(f"Shock Normals - {date}", fontsize=14)
    fig.subplots_adjust(top=0.9)
    plt.show()

In [18]:
def _normal_vector_cell(n, decimals=6):
    return f"[{n[0]:.{decimals}f}, {n[1]:.{decimals}f}, {n[2]:.{decimals}f}]"


def build_shock_normal_table(date):
    """Build per-satellite shock normal table with vector/theta/phi for each method."""
    dfs_local = get_dfs(date)
    params = _get_date_params(date)
    planar_data, spherical_data = get_normals_for_date(date)

    sphere_normals = {}
    if spherical_data is not None and spherical_data.get("status") == "ok":
        sphere_normals = spherical_data.get("normals", {})

    n_surface = None
    if planar_data is not None and planar_data.get("status") == "ok":
        n_surface = np.array(planar_data["normal"], dtype=float)

    excluded_sats = {"themis_c", "stereo"}
    rows = []

    for sat0 in sorted(dfs_local.keys()):
        sat_key = str(sat0).lower()
        if sat_key in excluded_sats or sat_key.startswith("stereo"):
            continue

        p = _get_sat_params(params, sat_key)
        if p is None:
            continue

        pos = get_sat_position_re(dfs_local[sat0], p["t0"], sat_name=sat0)
        if pos is None:
            continue

        methods = {
            "mx3": compute_mx3_normal(
                dfs_local[sat0], p["t0"], p["dt0_u"], p["dt1_u"], p["dt0_d"], p["dt1_d"]
            ),
            "mc": compute_mc_normal(
                dfs_local[sat0], p["t0"], p["dt0_u"], p["dt1_u"], p["dt0_d"], p["dt1_d"]
            ),
            "sphere": (
                np.array(sphere_normals[sat0], dtype=float)
                if sat0 in sphere_normals
                else None
            ),
            "surface": n_surface,
        }

        row = {
            "satellite": sat0,
            "date": date,
            "x": float(pos["x"]),
            "y": float(pos["y"]),
            "z": float(pos["z"]),
        }

        for name, n in methods.items():
            if n is None:
                row[f"{name}_n"] = np.nan
                row[f"{name}_theta"] = np.nan
                row[f"{name}_phi"] = np.nan
                continue

            n = np.array(n, dtype=float)
            if (not np.isfinite(n).all()) or np.linalg.norm(n) < 1e-12:
                row[f"{name}_n"] = np.nan
                row[f"{name}_theta"] = np.nan
                row[f"{name}_phi"] = np.nan
                continue

            theta, phi = theta_phi(n)
            row[f"{name}_n"] = _normal_vector_cell(n)
            row[f"{name}_theta"] = float(theta)
            row[f"{name}_phi"] = float(phi)

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows).set_index("satellite")


def print_shock_normal_coords(date):
    """Print per-satellite normal vectors and (theta, phi) angles by method."""
    out = build_shock_normal_table(date)
    if out.empty:
        print(f"No satellite normals available for {date}")
        return out
    display(out)
    return out


if "normals_date_dd" in globals() and normals_date_dd.options:
    _coords_df = print_shock_normal_coords(normals_date_dd.value)
elif shock_normals_data is not None:
    _fallback_date = next(
        (
            d
            for d, v in shock_normals_data.items()
            if isinstance(v, dict) and v.get("spherical", {}).get("status") == "ok"
        ),
        None,
    )
    if _fallback_date is not None:
        _coords_df = print_shock_normal_coords(_fallback_date)
    else:
        _coords_df = pd.DataFrame()
else:
    _coords_df = pd.DataFrame()

,date,x,y,z,mx3_n,mx3_theta,mx3_phi,mc_n,mc_theta,mc_phi,sphere_n,sphere_theta,sphere_phi,surface_n,surface_theta,surface_phi
satellite,,,,,,,,,,,,,,,,
ace,2022-01-16 19-09-00,223.340232,36.537002,-4.269207,"[-0.850946, 0.040657, -0.523677]",211.579242,177.264580,"[-0.586777, -0.442446, 0.678185]",137.298046,217.017330,"[0.555885, 0.315706, -0.768974]",230.261860,150.406313,"[-0.951200, -0.083458, 0.297076]",162.717933,185.014274
dscovr,2022-01-16 19-09-00,222.869566,-40.473401,-25.417224,"[-0.918659, -0.220100, -0.328058]",199.150968,193.473405,"[-0.646804, -0.292362, 0.704393]",135.219480,204.323436,"[0.552817, 0.106491, -0.826470]",235.737803,169.096481,"[-0.951200, -0.083458, 0.297076]",162.717933,185.014274
mms1,2022-01-16 19-09-00,22.731342,6.814397,0.848174,NaN,NaN,NaN,"[-0.726000, -0.681091, -0.095075]",185.455633,223.171977,"[-0.575026, 0.264976, -0.774037]",230.717800,155.259425,"[-0.951200, -0.083458, 0.297076]",162.717933,185.014274
wind,2022-01-16 19-09-00,254.602051,32.386753,12.675616,"[-0.499713, -0.259458, -0.826419]",235.732666,207.438949,"[-0.726775, -0.578315, 0.370608]",158.246912,218.510211,"[0.618729, 0.305553, -0.723748]",226.364809,153.718032,"[-0.951200, -0.083458, 0.297076]",162.717933,185.014274


# Shock Normals

Visualize shock normals calculated from the Wolfram JSON file, showing both spherical and planar normal methods in 2D projections and polar coordinate plots.

In [19]:
# Dates that have a valid sphere fit
if shock_normals_data is not None:
    _ok_dates = [
        d
        for d, v in shock_normals_data.items()
        if v.get("spherical", {}).get("status") == "ok"
    ]
else:
    _ok_dates = []

normals_date_dd = widgets.Dropdown(options=_ok_dates, description="Date:")
normals_out = widgets.Output()


def _on_normals_date(change):
    normals_out.clear_output(wait=True)
    with normals_out:
        plot_shock_normals(change["new"])


normals_date_dd.observe(_on_normals_date, names="value")

display(normals_date_dd, normals_out)

# Trigger initial plot only if there are dates available
if _ok_dates:
    with normals_out:
        plot_shock_normals(normals_date_dd.value)

Dropdown(description='Date:', options=('2022-01-16 19-09-00', '2023-02-26 19-23-00', '2024-04-29 18-31-00', '2…

Output()

# Shock Overview (Magnitudes only)

Multi-satellite stacked plot of |B| and |V| around the shock time, one row per satellite.

In [20]:
import matplotlib.dates as mdates


def _compute_B_mag(df_plot):
    """Return |B| series from whichever columns are available."""
    if "B" in df_plot.columns:
        return df_plot["B"]
    for cols in [
        ("B_X_GSE", "B_Y_GSE", "B_Z_GSE"),
        ("B_X_HGE", "B_Y_HGE", "B_Z_HGE"),
        ("B_x", "B_y", "B_z"),
    ]:
        if all(c in df_plot.columns for c in cols):
            return np.sqrt(df_plot[cols[0]] ** 2 + df_plot[cols[1]] ** 2 + df_plot[cols[2]] ** 2)
    return None


def _compute_V_mag(df_plot):
    """Return |V| series from whichever columns are available."""
    if "V" in df_plot.columns:
        return df_plot["V"]
    if "v" in df_plot.columns:
        return df_plot["v"]
    if "Proton_V_moment" in df_plot.columns:
        return df_plot["Proton_V_moment"]
    for cols in [
        ("V_X_GSE", "V_Y_GSE", "V_Z_GSE"),
        ("V_X_HGE", "V_Y_HGE", "V_Z_HGE"),
        ("v_x", "v_y", "v_z"),
        ("Proton_VX_moment", "Proton_VY_moment", "Proton_VZ_moment"),
    ]:
        if all(c in df_plot.columns for c in cols):
            return np.sqrt(df_plot[cols[0]] ** 2 + df_plot[cols[1]] ** 2 + df_plot[cols[2]] ** 2)
    return None


def plot_shock_overview(date):
    """Shock-Plot style stacked figure: one row per satellite, |B| (left) and |V| (right)."""
    dfs_local = get_dfs(date)
    params = _get_date_params(date)

    sats = []
    for sat0 in sorted(dfs_local.keys()):
        p = _get_sat_params(params, sat0)
        if p is not None:
            # if sat0 not in ['stereo','themis_c']:
            sats.append(sat0)

    if not sats:
        print(f"No satellites with parameters for {date}")
        return

    n = len(sats)
    fig, axes = plt.subplots(nrows=n, figsize=(15, 2.5 * n), sharex=False)
    if n == 1:
        axes = [axes]

    for i, sat0 in enumerate(sats):
        ax = axes[i]
        p = params[sat0] if sat0 in params else _get_sat_params(params, sat0)
        df = dfs_local[sat0]

        t0 = _parse_t0(p["t0"])
        if t0 is None:
            continue

        # Ensure timezone consistency
        if len(df.index) > 0:
            if df.index[0].tz is not None and t0.tz is None:
                t0 = t0.tz_localize("UTC")
            elif df.index[0].tz is None and t0.tz is not None:
                t0 = t0.tz_localize(None)

        dt0_u = int(p.get("dt0_u", DEFAULT_DTS["dt0_u"]))
        dt1_u = int(p.get("dt1_u", DEFAULT_DTS["dt1_u"]))
        dt0_d = int(p.get("dt0_d", DEFAULT_DTS["dt0_d"]))
        dt1_d = int(p.get("dt1_d", DEFAULT_DTS["dt1_d"]))

        # ±40 min window
        start = t0 - pd.Timedelta(minutes=40)
        end = t0 + pd.Timedelta(minutes=40)
        df_plot = df[start:end]
        if df_plot.empty:
            df_plot = df

        marker = "-" if sat0 in high_mag_density else "-o"
        plt.rcParams["lines.markersize"] = 2.5

        # |B| on left axis
        b_mag = _compute_B_mag(df_plot)
        if b_mag is not None:
            b_clean = b_mag.dropna()
            ax.plot(b_clean.index, b_clean, marker, color="tab:blue", label="|B|")
        ax.set_ylabel("|B| (nT)", color="tab:blue")
        ax.tick_params(axis="y", labelcolor="tab:blue")

        # |V| on right axis
        ax_v = ax.twinx()
        marker_v = "-" if sat0 in high_v_density else "-o"
        plt.rcParams["lines.markersize"] = 3.5
        v_mag = _compute_V_mag(df_plot)
        if v_mag is not None:
            v_clean = v_mag.dropna()
            ax_v.plot(v_clean.index, v_clean, marker_v, color="tab:red", label="|V|")
        ax_v.set_ylabel("|V| (km/s)", color="tab:red")
        ax_v.tick_params(axis="y", labelcolor="tab:red")

        # Reference lines (same colors as in the big plot)
        _draw_reference_lines(ax, t0, dt0_u, dt1_u, dt0_d, dt1_d)

        ax.set_xlim(df_plot.index[0], df_plot.index[-1])

        # Satellite label on the left
        ax.set_title(sat0.upper(), rotation="vertical", x=-0.07, y=0.3, fontsize=11)

        # Combine legends
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax_v.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=12)

        # Tick formatting
        ax.minorticks_on()
        ax.xaxis.set_major_locator(mdates.MinuteLocator(range(0, 120, 10)))
        ax.grid(visible=True, which="major", axis="x", linewidth=1.0)
        ax.grid(visible=True, which="minor", axis="x", linewidth=0.3)
        loc = ax.xaxis.get_major_locator()
        fmt = mdates.ConciseDateFormatter(loc)
        fmt.offset_formats = [""] * 6
        ax.xaxis.set_major_formatter(fmt)
        ax.tick_params(axis='both', which='major', labelsize=12)
    
    fig.suptitle(f"Shock Overview — {date}", fontsize=13)
    fig.tight_layout()
    fig.subplots_adjust(top=0.94)
    plt.show()


# --- Dropdown widget ---
_overview_dates = list_shock_dates()
overview_date_dd = widgets.Dropdown(options=_overview_dates, description="Date:")
overview_out = widgets.Output()


def _on_overview_date(change):
    overview_out.clear_output(wait=True)
    with overview_out:
        plot_shock_overview(change["new"])


overview_date_dd.observe(_on_overview_date, names="value")

display(overview_date_dd, overview_out)

if _overview_dates:
    with overview_out:
        plot_shock_overview(overview_date_dd.value)

Dropdown(description='Date:', options=('2022-01-16 19-09-00', '2023-02-26 19-23-00', '2023-07-16 19-20-00', '2…

Output()

## MX3 Export for Integration Tests

Use `export_mx3_normals_for_events(...)` to export notebook-computed MX3 normals
for a fixed set of events/satellites.

In [ ]:
def export_mx3_normals_for_events(event_specs, output_path=Path("mx3_results.json"), require_all=False):
    """Export MX3 normals computed by this notebook for integration tests."""
    results = {}
    missing = []

    for event, sats in event_specs.items():
        dfs = get_dfs(event)
        event_out = {}

        for sat, spec in sats.items():
            if sat not in dfs:
                event_out[sat] = None
                missing.append((event, sat, "missing_satellite_file"))
                continue

            n = compute_mx3_normal(
                dfs[sat],
                spec["t0"],
                spec["dt0_u"],
                spec["dt1_u"],
                spec["dt0_d"],
                spec["dt1_d"],
            )
            if n is None:
                event_out[sat] = None
                missing.append((event, sat, "mx3_unavailable"))
            else:
                event_out[sat] = [float(x) for x in np.asarray(n, dtype=float)]

        results[event] = event_out

    output_path = Path(output_path)
    output_path.write_text(json.dumps(results, indent=2, sort_keys=True))

    if require_all and missing:
        raise ValueError(f"Missing MX3 outputs: {missing}")

    return results
